# 03c — Precipitación diaria CHIRPS v3 (final, sat) para Piura

Lee de los COG diarios de la CHC solo el recuadro del departamento de Piura (polígonos GADM + 0,05°; un bloque de
512 × 512 del archivo, unos 112 KiB por día) y guarda la ventana de cada día, tal cual, en
`data/bronze/chirps/v3_sat/final/AAAA/`. Cada corrida deja un manifiesto en `data/bronze/chirps/v3_sat/`.

- **Reanudación:** cada día es un archivo; se escribe con nombre temporal y se renombra al terminar. Si se corta,
  al volver a correr la celda B se saltan los días ya guardados. Nunca se sobrescribe.
- **Límite de tasa:** el servidor responde HTTP 429 tras una ráfaga y no publica su política. Los primeros 50 días
  miden la tasa real; la pausa se ajusta y, si la estimación corregida supera 16 h, la corrida se detiene.
- **En la V3 la lluvia es solo CHIRPS v3** (la de Open-Meteo queda para la V2).
- **Fuente y licencia** (https://www.chc.ucsb.edu/data/chirps3): CHIRPS3 está en dominio público y con licencia
  Creative Commons Attribution 4.0 International. Citar el repositorio (https://doi.org/10.15780/G2JQ0P, con la
  fecha de acceso) y el artículo de Funk et al. (2026), *Sci Data* 13, 718
  (https://doi.org/10.1038/s41597-026-07096-4), con la lista de autores de la página oficial.


In [ ]:
import pandas as pd
from datetime import date

from denguekay.config.paths import BRONZE_CHIRPS, BRONZE_CHIRPS_FINAL, DISTRICT_COORDS
from denguekay.data.ingestion.chirps import check_sample_days, download_final

distritos = pd.read_csv(DISTRICT_COORDS)   # los mismos centroides que Open-Meteo
print(len(distritos), "distritos |", BRONZE_CHIRPS_FINAL)


## Correr desde el worktree (junction de `data/bronze/chirps`)

Si la descarga se corre desde un worktree (por ejemplo `..\Denguekay-feat-chirps`, con su propio `.venv`), su
carpeta `data/` no es la de la carpeta principal: solo trae lo versionado (`data/reference/`, los `.gitkeep`). Para
que el 03c escriba en el `data/` real, se enlaza **solo** `data/bronze/chirps` (ignorado por git y sin archivos
versionados), nunca la carpeta `data` entera.

**Crear** (PowerShell, desde la raíz del worktree):

```powershell
New-Item -ItemType Directory -Force C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\data\bronze\chirps
New-Item -ItemType Junction -Path data\bronze\chirps -Target C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\data\bronze\chirps
git status --short   # no debe mostrar nada nuevo
```

**Quitar** (borra solo el enlace; los datos quedan en la carpeta principal):

```powershell
cmd /c rmdir data\bronze\chirps
```

Nunca uses `Remove-Item -Recurse` (ni `rm -r`) sobre el enlace: puede seguirlo y borrar los datos de la carpeta
principal. Comprueba antes con `Get-Item data\bronze\chirps | Select-Object LinkType, Target` que es un
`Junction`.


## A · Verificación al inicio: un día de 2000 y uno de 2025

Mismo −9999 en el mar (y misma máscara), mismo tamaño de píxel (0,05°), valores ≥ 0 o −9999, y ningún centroide
fuera de la ventana o sin dato. Si algo difiere, la celda se detiene y no se descarga nada.


In [ ]:
verificacion = check_sample_days([date(2000, 1, 15), date(2025, 1, 15)], distritos)
if not verificacion["ok"]:
    raise RuntimeError(f"La verificación inicial falló: {verificacion['problemas']}")
verificacion["dias"]


## B · Historia 2000 → hoy por FTP (final sat, sin cuenta)

El final de CHIRPS v3 se baja del FTP anónimo de la CHC (`ftp.chc.ucsb.edu`), un archivo global por día (unos 11 MB,
se guarda solo la ventana de Piura en `data/bronze/chirps/v3_sat/final_ftp/AAAA/`). No hace falta cuenta de Earth
Engine. Son unos 9 500 días: con la pausa de 1 s y la transferencia, del orden de 10 h (depende de la conexión). Se
detiene en el primer error; al volver a correr la celda, los días ya guardados se comparan y no se reescriben.

Alternativa por HTTPS (COG, solo la ventana): `download_final(...)` de `denguekay.data.ingestion.chirps`; el servidor
limita la tasa (HTTP 429) y la celda calibra el ritmo.


In [ ]:
from denguekay.pipelines.chirps_prelim import ftp

ftp("final", date(2000, 1, 1))   # hasta el último día final publicado


## C · Preliminar reciente y silver semanal

El preliminar cubre las semanas sin final completo. El silver se escribe en un archivo NUEVO de `data/silver/` (nunca
sobre el vigente); por defecto usa solo el final por FTP. `--con-earth-engine` (o `earth_engine=True`) reconstruye el
silver con el que se entrenó la v3, que tomó el final del snapshot de Earth Engine (`ee_final`, en los Releases). En
los 240 días que se solapan (enero a abril de 2025 y 2026), las medias diarias por polígono del FTP y de Earth Engine
difieren como máximo 0,15 mm (media 0,003 mm) y las sumas semanales como máximo 0,34 mm.


In [ ]:
ESCRIBIR_SILVER = False
if ESCRIBIR_SILVER:
    from denguekay.pipelines.chirps_prelim import silver_ftp

    ultimo_final = date.today()
    ftp("prelim", date(ultimo_final.year, 1, 1))
    silver_ftp(f"precip_chirps_v3_semanal_desde_2000_{date.today():%Y%m%d}.csv")
